# Apigee Template: REST-AI-Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gcp-samples/apigee-template-repository/blob/main/notebooks/REST-AI-Embeddings.ipynb)

**Template Name:** `REST-AI-Embeddings`  
**Description:** Unified OpenAI-compatible vector embeddings API proxy (`/v1/embeddings`) providing embeddings generation across Google Cloud Vertex AI (e.g. `text-embedding-004`, `text-embedding-005`) and OpenAI (`text-embedding-3-small`, `text-embedding-3-large`) with automated format mediation, IAM token generation, and token analytics.

### Key Capabilities:
- **OpenAI-Compatible `/v1/embeddings` Protocol:** Accept standard OpenAI embedding requests (`{"model": "...", "input": "..."}`).
- **Protocol & Format Mediation:** Bidirectional translation between OpenAI embeddings schema and Google Cloud Vertex AI `embedContent` format.
- **Multi-Provider Routing:** Routes requests dynamically to Google Cloud Vertex AI or OpenAI based on the requested model.
- **Automated IAM & Security:** Automatically attaches Google Cloud OAuth tokens via `AM-SetGoogleToken`, removing the need for callers to manage GCP credentials directly.
- **Token Usage Analytics:** Intercepts response token counts and populates Apigee Data Collectors (`dc_ai_prompt_token_count`, `dc_ai_total_token_count`, `dc_ai_model`).

### Documentation & References:
- [OpenAI Embeddings API Reference](https://platform.openai.com/docs/api-reference/embeddings)
- [Google Cloud Vertex AI Embeddings API](https://cloud.google.com/vertex-ai/generative-ai/docs/embeddings/get-text-embeddings)
- [Apigee Feature Templater (aft) GitHub](https://github.com/apigee/apigee-templater)
- [Apigee Data Collectors Documentation](https://cloud.google.com/apigee/docs/api-platform/analytics/data-collectors)

---

## 1. Configuration & Authentication

Specify your Google Cloud Project ID (Apigee Organization), Apigee environment, and optional API keys.

In [ ]:
# @title 1. Configuration & Authentication
import os

# @markdown Enter your Google Cloud Project ID (Apigee Organization):
GOOGLE_CLOUD_PROJECT = "your_apigee_org"  # @param {type:"string"}
APIGEE_ENV = "dev"  # @param {type:"string"}
OPENAI_API_KEY = ""  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
os.environ["APIGEE_ORG"] = GOOGLE_CLOUD_PROJECT
os.environ["APIGEE_ENV"] = APIGEE_ENV
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["APIGEE_SA"] = f"apigee-service@{GOOGLE_CLOUD_PROJECT}.iam.gserviceaccount.com"

try:
    from google.colab import auth
    auth.authenticate_user()
    print(f"Authenticated with Google Cloud for project: {GOOGLE_CLOUD_PROJECT}")
except ImportError:
    print("Running outside Google Colab.")


## 2. Setup Tools, Initialize Resources & Deploy Template

Installs `aft`, runs `sh/initialize.sh` (sets up service account, IAM permissions, data collectors, and custom reports), resolves `APIGEE_HOST`, and deploys `REST-AI-Embeddings.yaml`.

In [ ]:
# @title 2. Setup, Initialize & Deploy Template
import os
import subprocess

# 1. Install Apigee Feature Templater (aft) CLI if needed
!which aft >/dev/null 2>&1 || curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh

# 2. Download template and initialize script if running standalone
REPO_RAW = "https://raw.githubusercontent.com/gcp-samples/apigee-template-repository/main"
TEMPLATE_FILE = "REST-AI-Embeddings.yaml" if os.path.exists("REST-AI-Embeddings.yaml") else "templates/REST-AI-Embeddings.yaml" if os.path.exists("templates/REST-AI-Embeddings.yaml") else "REST-AI-Embeddings.yaml"
os.environ["TEMPLATE_FILE"] = TEMPLATE_FILE

if not os.path.exists(TEMPLATE_FILE):
    !curl -fsSL -O {REPO_RAW}/templates/REST-AI-Embeddings.yaml

if not os.path.exists("sh/initialize.sh"):
    !mkdir -p sh && curl -fsSL -o sh/initialize.sh {REPO_RAW}/sh/initialize.sh

# 3. Run initialization script
!bash sh/initialize.sh

# 4. Resolve APIGEE_HOST using aft describe
cmd = 'aft describe --project "$GOOGLE_CLOUD_PROJECT" -f json | jq --raw-output ".environmentGroups[] | select(any(.attachments[]; .environment == \"$APIGEE_ENV\")) | .hostnames[0]"'
try:
    host = subprocess.check_output(cmd, shell=True, text=True).strip()
    if host and host != "null":
        os.environ["APIGEE_HOST"] = host
        print(f"APIGEE_HOST resolved to: {host}")
except Exception as e:
    print(f"Could not automatically resolve APIGEE_HOST via aft: {e}")

# 5. Deploy template with aft
!aft "$TEMPLATE_FILE" \
  --project="$GOOGLE_CLOUD_PROJECT" \
  --env="$APIGEE_ENV" \
  --sa="$APIGEE_SA"


## 3. Test Embeddings API via APIGEE_HOST

Send requests to `https://${APIGEE_HOST}/v1/embeddings` using standard OpenAI request payloads.

In [ ]:
# @title Setup Test Client & APIGEE_HOST
import os
import json
import requests
import subprocess

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "")
APIGEE_ENV = os.getenv("APIGEE_ENV", "dev")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Determine APIGEE_HOST
APIGEE_HOST = os.getenv("APIGEE_HOST")
if not APIGEE_HOST or APIGEE_HOST == "null":
    APIGEE_HOST = f"{PROJECT_ID}-{APIGEE_ENV}.apigee.net"
    os.environ["APIGEE_HOST"] = APIGEE_HOST

# Retrieve caller access token if available
try:
    gcp_token = subprocess.check_output(["gcloud", "auth", "application-default", "print-access-token"], text=True).strip()
except Exception:
    try:
        gcp_token = subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()
    except Exception:
        gcp_token = ""

print(f"APIGEE_HOST: {APIGEE_HOST}")
print(f"Embeddings Endpoint: https://{APIGEE_HOST}/v1/embeddings")

def create_embeddings(model: str, input_text):
    """
    Calls the Apigee /v1/embeddings endpoint with OpenAI-compatible payload.
    """
    url = f"https://{APIGEE_HOST}/v1/embeddings"
    headers = {"Content-Type": "application/json"}
    if gcp_token:
        headers["Authorization"] = f"Bearer {gcp_token}"
    if OPENAI_API_KEY:
        headers["x-api-key"] = OPENAI_API_KEY

    payload = {
        "model": model,
        "input": input_text
    }

    preview = input_text if isinstance(input_text, str) else f"{len(input_text)} items"
    print(f"\n---> Sending [{model}] embeddings request for: {preview}...")
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        print(f"HTTP Status: {response.status_code}")
        try:
            data = response.json()
            items = data.get("data", [])
            print(f"Response Object: {data.get('object', 'unknown')}")
            print(f"Embeddings Count: {len(items)}")
            if items:
                vec = items[0].get("embedding", [])
                print(f"Vector Dimensions: {len(vec)}")
                print(f"Sample Values (first 5 floats): {vec[:5]}")
            usage = data.get("usage", {})
            if usage:
                print(f"Usage Tokens: prompt={usage.get('prompt_tokens', 'N/A')}, total={usage.get('total_tokens', 'N/A')}")
            return data
        except Exception:
            print(response.text)
            return None
    except Exception as e:
        print("Request error:", e)
        return None


In [ ]:
# @title Test 1: Generate Embedding for a Single Text String
# Proxies request to Vertex AI text-embedding-004 and maps response to OpenAI format
res1 = create_embeddings(
    model="text-embedding-004",
    input_text="Apigee provides enterprise governance, rate limiting, and security for AI APIs."
)


In [ ]:
# @title Test 2: Semantic Similarity with Generated Embeddings
# Computes cosine similarity between two sentences using vector embeddings
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    return dot / (norm_a * norm_b) if (norm_a and norm_b) else 0.0

text_a = "Artificial intelligence models generate text and code."
text_b = "Machine learning LLMs produce natural language and programming scripts."
text_c = "A delicious recipe for homemade sourdough pizza."

emb_a = create_embeddings("text-embedding-004", text_a)
emb_b = create_embeddings("text-embedding-004", text_b)
emb_c = create_embeddings("text-embedding-004", text_c)

if emb_a and emb_b and emb_c:
    vec_a = emb_a["data"][0]["embedding"]
    vec_b = emb_b["data"][0]["embedding"]
    vec_c = emb_c["data"][0]["embedding"]

    sim_ab = cosine_similarity(vec_a, vec_b)
    sim_ac = cosine_similarity(vec_a, vec_c)

    print("\n--- Semantic Similarity Results ---")
    print(f"Similarity (Related topics: AI vs LLMs):      {sim_ab:.4f}")
    print(f"Similarity (Unrelated: AI vs Sourdough Pizza): {sim_ac:.4f}")


## 4. Verify Apigee Analytics & Token Data Collection

Each embeddings request processed by the gateway extracts token counts and populates Apigee Data Collectors:
- `dc_ai_prompt_token_count`
- `dc_ai_total_token_count`
- `dc_ai_model`

View these analytics under **Analytics > Custom Reports** in the [Apigee Console](https://console.cloud.google.com/apigee).